# Phase 2 V5 Demo Inference

Upload `phase2_v5_model_bundle.pkl` to Colab, then run these cells.

This notebook is demo-only. It loads the trained model bundle and runs single-prompt or batch-prompt inference.

In [ ]:
!pip -q install sentence-transformers xgboost

import json
import pickle
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_BUNDLE_PATH = '/content/phase2_v5_model_bundle.pkl'

with open(MODEL_BUNDLE_PATH, 'rb') as f:
    bundle = pickle.load(f)

embedding_model = SentenceTransformer(bundle['embedding_model_name'])
final_pca = bundle['pca']
final_scaler = bundle['scaler']
heads = bundle['heads']
label_encoders = bundle['label_encoders']
CLASS_TO_SCORE = bundle['class_to_score']
DIMENSION_LABELS = bundle['dimension_labels']
VALID_SCORES = bundle['valid_scores']
SCORE_COLS = bundle['score_cols']
DIRECT_WEIGHT = bundle.get('tier_blend_direct_weight', 0.72)
FORMULA_WEIGHT = bundle.get('tier_blend_formula_weight', 0.28)

print('Loaded model bundle.')
print('Heads:', list(heads.keys()))

In [ ]:
ARTIFACT_TERMS = ['csv', 'json', 'pdf', 'log', 'yaml', 'yml', 'xlsx', 'docx', 'transcript', 'diagram']
CLOUD_PROVIDERS = ['aws', 'azure', 'gcp', 'google cloud', 'oci']
SYSTEMS = ['salesforce', 'servicenow', 'jira', 'workday', 'sap', 'snowflake', 'databricks', 'okta', 'hubspot', 'github', 'gitlab']
FRAMEWORKS = ['itil', 'finops', 'togaf', 'owasp', 'dora', 'nist', 'hipaa', 'soc 2', 'soc2', 'gdpr', 'iso 27001', 'pci-dss', 'pci dss', 'cis benchmark', 'fhir']
VENDOR_TOOLS = sorted(set(CLOUD_PROVIDERS + SYSTEMS + ['openai', 'anthropic', 'bedrock', 'terraform', 'kubernetes', 'docker', 'jenkins', 'splunk', 'redis', 'prometheus', 'opentelemetry']))
DOMAIN_BUCKETS = {
    'cloud': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'budget', 'chargeback', 'showback'],
    'security': ['security', 'vulnerability', 'iam', 'zero trust', 'soc'],
    'devops': ['devops', 'ci/cd', 'pipeline', 'sre', 'deployment'],
    'data': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark', 'dbt'],
    'ai': ['ai', 'llm', 'genai', 'machine learning', 'model'],
    'hr': ['hr', 'employee', 'talent', 'workforce', 'recruiting', 'workday'],
    'supply': ['supply chain', 'inventory', 'procurement', 'logistics'],
}
D2_DOMAIN_TERMS = sorted(set(CLOUD_PROVIDERS + SYSTEMS + FRAMEWORKS + VENDOR_TOOLS + [
    'hl7', 'fhir', 'redis', 'ttl', 'cache stampede', 'prometheus', 'opentelemetry',
    'slo', 'burn-rate', 'active directory', 'ransomware', 'control tower', 'azure policy'
]))
D1_COMPLEXITY_TERMS = [
    'synthesize', 'strategic', 'operating model', 'tradeoffs', 'roadmap', 'governance',
    'cross-functional', 'executive', 'architecture', 'migration', 'rollout', 'risk',
    'alternatives', 'recommendation', 'phased', 'metrics'
]
SIMPLE_FACTUAL_PATTERNS = [r'^what is\b', r'^define\b', r'^explain .* in simple terms', r'one simple example']

RESEARCH_SIGNAL_KEYWORDS = {
    'market_research': ['market', 'industry', 'trend', 'tam', 'sam', 'som'],
    'competitive_analysis': ['competitor', 'competitive', 'benchmark', 'rival'],
    'regulatory_compliance': ['regulation', 'regulatory', 'compliance', 'gdpr', 'hipaa', 'sox', 'eu ai act'],
    'security': ['security', 'vulnerability', 'threat', 'risk', 'iam', 'zero trust'],
    'cloud_infrastructure': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'spend', 'budget', 'showback', 'chargeback'],
    'devops': ['ci/cd', 'pipeline', 'deployment', 'sre', 'devops', 'observability'],
    'data_engineering': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai_governance': ['ai governance', 'llm', 'model risk', 'genai', 'guardrail'],
    'system_integration': ['integration', 'api', 'webhook', 'middleware'],
    'supply_chain': ['supply chain', 'inventory', 'procurement', 'logistics'],
    'hr_tech': ['hr', 'employee', 'workforce', 'talent', 'recruiting'],
    'vendor_analysis': ['vendor', 'rfi', 'rfp', 'procurement'],
}

def term_present(text, term):
    return bool(re.search(rf'(?<![A-Za-z0-9_]){re.escape(term)}(?![A-Za-z0-9_])', text))

def count_terms_safe(text, terms):
    return sum(1 for term in terms if term_present(text, term))

def any_terms_safe(text, terms):
    return any(term_present(text, term) for term in terms)

def complexity_score_from_dims(dims):
    return round(dims['d1'] * 0.35 + dims['d2'] * 0.20 + dims['d3'] * 0.20 + dims['d4'] * 0.15 + dims['d5'] * 0.10, 4)

def tier_from_score(score):
    if score < 0.40:
        return 'T1'
    if score < 0.70:
        return 'T2'
    return 'T3'

def handcrafted_features(prompts):
    rows = []
    for prompt in prompts:
        text = str(prompt)
        lower = text.lower()
        words = re.findall(r'\b\w+\b', lower)
        unique_words = set(words)
        sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
        lines = [line for line in text.splitlines() if line.strip()]
        row = {}
        row['char_len'] = len(text)
        row['word_count'] = len(words)
        row['sentence_count'] = max(1, len(sentences))
        row['avg_word_len'] = float(np.mean([len(w) for w in words])) if words else 0.0
        row['unique_word_ratio'] = len(unique_words) / max(1, len(words))
        row['line_count'] = len(lines)
        row['has_attachment'] = int(any(phrase in lower for phrase in ['uploaded', 'attached', 'provided file', 'document below', 'context below', 'see below']))
        row['provided_artifact_count'] = count_terms_safe(lower, ARTIFACT_TERMS)
        row['large_context_signal'] = int(any(phrase in lower for phrase in ['across all', 'entire', 'all of our', 'company-wide', 'large context', 'full document']))
        row['multi_document_signal'] = int(any(phrase in lower for phrase in ['multiple', 'all the', 'each of the', 'various', 'several documents', 'set of files']))
        row['has_formal_deliverable'] = int(any_terms_safe(lower, ['report', 'brief', 'proposal', 'specification', 'whitepaper']) or 'requirements doc' in lower)
        row['has_report_package'] = int(any(phrase in lower for phrase in ['appendix', 'table of contents', 'risk register', 'executive summary', 'roadmap', 'implementation plan']))
        row['has_long_output_signal'] = int(any(phrase in lower for phrase in ['comprehensive', 'detailed', 'thorough', 'in-depth', 'end-to-end']))
        row['structured_section_count'] = count_terms_safe(lower, ['timeline', 'roadmap', 'assumptions', 'recommendations']) + sum(1 for phrase in ['executive summary', 'risk register', 'next steps', 'success metrics'] if phrase in lower)
        row['has_scope_words'] = int(any_terms_safe(lower, ['strategic', 'synthesize', 'governance']) or any(phrase in lower for phrase in ['cross-domain', 'enterprise-wide', 'multi-cloud']))
        row['action_verb_count'] = count_terms_safe(lower, ['build', 'design', 'evaluate', 'integrate', 'optimize', 'develop', 'assess', 'recommend', 'compare'])
        row['multi_stage_signal'] = int(bool(re.search(r'\bphase\b|\bstage\b|\bstep\s*1\b|\bmilestone\b|\bsequentially\b|\bfirst\b.*\bthen\b', lower)))
        row['has_compliance'] = int(any_terms_safe(lower, ['nist', 'hipaa', 'soc2', 'gdpr', 'compliance']) or any(phrase in lower for phrase in ['soc 2', 'iso 27001', 'pci-dss', 'pci dss']))
        row['cloud_providers_mentioned'] = count_terms_safe(lower, CLOUD_PROVIDERS)
        row['systems_mentioned'] = count_terms_safe(lower, SYSTEMS)
        row['domain_framework_count'] = count_terms_safe(lower, FRAMEWORKS)
        row['external_data_score'] = count_terms_safe(lower, ['analyst', 'latest', 'current']) + sum(1 for phrase in ['market research', 'industry report', 'third-party', 'external data'] if phrase in lower)
        row['has_time_reference'] = int(bool(re.search(r'\b20\d{2}\b|\bfy\d{2}\b|\bthis quarter\b|\blatest\b|\bcurrent\b|\brecent\b|\btoday\b|\bnow\b', lower)))
        row['vendor_tool_count'] = count_terms_safe(lower, VENDOR_TOOLS)
        row['has_market_terms'] = int(any_terms_safe(lower, ['competitor', 'benchmark']) or any(phrase in lower for phrase in ['market share', 'industry trend']) or any_terms_safe(lower, ['tam', 'sam', 'som']))
        row['has_cost_comparison'] = int(any_terms_safe(lower, ['pricing', 'tco', 'roi', 'showback', 'chargeback', 'cheapest']) or 'cost analysis' in lower)
        row['has_comparison'] = int(any_terms_safe(lower, ['compare', 'versus', 'tradeoff']) or any(phrase in lower for phrase in [' vs ', 'difference between']))
        row['stakeholder_mentions'] = count_terms_safe(lower, ['ceo', 'cto', 'cio', 'cfo', 'board', 'leadership', 'management', 'executive'])
        row['risk_language'] = count_terms_safe(lower, ['risk', 'threat', 'vulnerability', 'mitigation', 'breach', 'exposure', 'audit'])
        row['has_role_prompt'] = int(bool(re.search(r'\byou are\b|\bact as\b|\bassume the role\b', lower)))
        row['has_step_request'] = int(bool(re.search(r'\bstep[- ]by[- ]step\b|\bfirst\b.*\bthen\b|\bsequentially\b', lower)))
        row['has_chain_of_thought'] = int(bool(re.search(r'\bthink through\b|\breason about\b|\blet.s think\b|\bchain of thought\b|\bwalk me through\b', lower)))
        if '?' not in text:
            row['question_complexity'] = 0
        elif any(phrase in lower for phrase in ['what should', 'design a']) or any_terms_safe(lower, ['recommend', 'propose', 'strategy']):
            row['question_complexity'] = 3
        elif any_terms_safe(lower, ['why', 'how', 'compare', 'analyze', 'evaluate', 'assess']):
            row['question_complexity'] = 2
        else:
            row['question_complexity'] = 1
        row['multi_domain_count'] = sum(1 for bucket_terms in DOMAIN_BUCKETS.values() if any_terms_safe(lower, bucket_terms))
        row['has_code_block'] = int('```' in text)
        row['has_output_format'] = int(bool(re.search(r'\bin json\b|\bas a table\b|\bformat as\b|\bcsv output\b|\bin yaml\b|\bas markdown\b|\bstrict yaml\b|\bstrict json\b', lower)))
        row['has_creative_language'] = int(any_terms_safe(lower, ['imagine', 'creative', 'story', 'compose', 'fictional']) or 'write a' in lower)
        row['has_classification_request'] = int(any_terms_safe(lower, ['classify', 'categorize', 'label']) or any(phrase in lower for phrase in ['which category', 'sort into']))
        row['enumeration_signal'] = int(bool(re.search(r'\blist\b|\btop \d+\b|\benumerate\b|\bbullet point\b|\brank\b', lower)))
        row['d1_strategic_signal_count'] = count_terms_safe(lower, D1_COMPLEXITY_TERMS)
        row['d1_simple_factual_signal'] = int(any(re.search(pattern, lower) for pattern in SIMPLE_FACTUAL_PATTERNS))
        row['d1_multi_constraint_count'] = count_terms_safe(lower, ['include', 'cover', 'consider', 'account for', 'must'])
        row['d1_solution_design_signal'] = int(any_terms_safe(lower, ['design', 'architect', 'plan', 'strategy', 'roadmap']))
        row['d2_domain_term_count'] = count_terms_safe(lower, D2_DOMAIN_TERMS)
        row['d2_acronym_count'] = len(re.findall(r'\b[A-Z]{2,6}\b', text))
        row['d2_vendor_or_framework_signal'] = int(row['vendor_tool_count'] > 0 or row['domain_framework_count'] > 0)
        row['d2_generic_prompt_signal'] = int(row['d2_domain_term_count'] == 0 and row['cloud_providers_mentioned'] == 0 and row['systems_mentioned'] == 0)
        row['phrasing_explicit'] = 0
        row['phrasing_implicit'] = 0
        row['phrasing_vague'] = 0
        rows.append(row)
    return pd.DataFrame(rows).fillna(0)

def extract_research_signals(prompt, d4_score):
    if d4_score <= 0:
        return []
    text = str(prompt).lower()
    signals = []
    for signal, keywords in RESEARCH_SIGNAL_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            signals.append(signal)
    return signals if signals else ['external_research']

def build_features_for_prompts(prompts):
    prompts = [str(p) for p in prompts]
    embeddings = embedding_model.encode(prompts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    hand = handcrafted_features(prompts)
    raw = np.hstack([final_pca.transform(embeddings), hand.values])
    return final_scaler.transform(raw)

def predict_prompt(prompt):
    X_one = build_features_for_prompts([prompt])
    direct_tier_proba = heads['tier'].predict_proba(X_one)
    tier_class = int(np.argmax(direct_tier_proba, axis=1)[0])
    direct_tier = label_encoders['tier'].inverse_transform([tier_class])[0]

    X_one_aug = np.hstack([X_one, np.array([[tier_class]])])
    dims = {}
    for col in SCORE_COLS:
        pred_class = int(heads[col].predict(X_one_aug)[0])
        dims[col] = CLASS_TO_SCORE[pred_class]

    score = complexity_score_from_dims(dims)
    formula_tier = tier_from_score(score)
    formula_onehot = np.zeros_like(direct_tier_proba)
    formula_onehot[0, label_encoders['tier'].transform([formula_tier])[0]] = 1.0
    blended_proba = DIRECT_WEIGHT * direct_tier_proba + FORMULA_WEIGHT * formula_onehot
    final_tier = label_encoders['tier'].inverse_transform([int(np.argmax(blended_proba, axis=1)[0])])[0]

    intent = label_encoders['intent'].inverse_transform(heads['intent'].predict(X_one))[0]
    task_type = label_encoders['task_type'].inverse_transform(heads['task_type'].predict(X_one))[0]
    reasoning_chain = bool(int(heads['reasoning_chain_detected'].predict(X_one)[0]))
    tier_confidence = float(np.max(blended_proba))

    result = {}
    for col in SCORE_COLS:
        result[col] = dims[col]
        result[f'{col}_label'] = DIMENSION_LABELS[col]
    result.update({
        'complexity_score': score,
        'tier': final_tier,
        'direct_tier': direct_tier,
        'formula_tier': formula_tier,
        'intent': intent,
        'task_type': task_type,
        'reasoning_chain_detected': reasoning_chain,
        'research_signals': extract_research_signals(prompt, dims['d4']),
        'confidence': round(float(np.mean([tier_confidence, np.max(heads['intent'].predict_proba(X_one)[0]), np.max(heads['task_type'].predict_proba(X_one)[0]), np.max(heads['reasoning_chain_detected'].predict_proba(X_one)[0])])), 4),
        'tier_confidence': round(tier_confidence, 4),
    })
    return result

In [ ]:
prompt = input('Enter a prompt: ')
result = predict_prompt(prompt)
print(json.dumps(result, indent=2))

In [ ]:
prompts = [
    'What is a webhook?',
    'Write a Python function that validates a JSON payload and returns structured errors.',
    'Compare AWS, Azure, and GCP pricing for a three-year GenAI platform roadmap and recommend an architecture.',
    'Summarize the attached incident report into executive bullets, root cause, impact, and next actions.',
]

rows = []
for prompt in prompts:
    result = predict_prompt(prompt)
    rows.append({
        'prompt': prompt,
        'tier': result['tier'],
        'tier_confidence': result['tier_confidence'],
        'intent': result['intent'],
        'task_type': result['task_type'],
        'complexity_score': result['complexity_score'],
        'd1': result['d1'],
        'd2': result['d2'],
        'd3': result['d3'],
        'd4': result['d4'],
        'd5': result['d5'],
    })

pd.DataFrame(rows)